# 00 - Project Setup

Run this once before the pipeline. It:
1. Creates the Unity Catalog and schemas
2. Creates the `source_files` volume for the source CSV files
3. Checks that both files were uploaded to the volume

**Safe to re-run.**

In [0]:
from configs.etl_config import (
    CATALOG,
    SCHEMAS,
    SOURCE_VOLUME_SCHEMA,
    SOURCE_VOLUME_NAME,
    SOURCE_VOLUME_PATH,
    CROP_FILE_NAME,
    WEATHER_FILE_NAME,
)

**1. Create catalog, schemas, and volume**

In [0]:
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')

for schema in SCHEMAS:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}')
    print(f'Schema ready: {CATALOG}.{schema}')

spark.sql(f'CREATE VOLUME IF NOT EXISTS {CATALOG}.{SOURCE_VOLUME_SCHEMA}.{SOURCE_VOLUME_NAME}')
print(f'Volume ready: {CATALOG}.{SOURCE_VOLUME_SCHEMA}.{SOURCE_VOLUME_NAME}')

**2. Check the source files**

Upload `crop_trend.csv` and `province_weather_quarterly.csv` to the volume
(Catalog -> `ph_crop_climate` -> `landing` -> `source_files` -> Upload), then re-run this cell.

In [0]:
files_in_volume = [file_info.name for file_info in dbutils.fs.ls(SOURCE_VOLUME_PATH)]
missing_files = []

for file_name in [CROP_FILE_NAME, WEATHER_FILE_NAME]:
    if file_name in files_in_volume:
        print(f'[X] {file_name}')
    else:
        print(f'[ ] {file_name}')
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(f'Upload the following files to {SOURCE_VOLUME_PATH}: {", ".join(missing_files)}')